## Modelling of pion decay 

THe production of $\mu+$ from SM events is proceeded via four channels namely, CC_numu, CC_nuebar, NC_numu, NC_nuebar where CC and NC stands for charged and neutral current respectively. During hadronization,light mesons such as pions and kaons are also produced. These light mesons can further decay to $\mu^+$ + $\nu_{\mu}$. So modelling such events are important to study the effect it has on the background.

EXTRA NOTE: I have created an event file with only one event to study the code. Also, currently I have only included pion decay ( will add kaon once everything is confirmed)

## Load Libraries

In [1]:
import pandas as pd
import numpy as np
import math
import os
from skhep.math.vectors import LorentzVector, Vector3D

## Load Example Event

Sample data for 3 events, each with pions and kaons. 

In [3]:
data = [
    # Event 1
    [1,0, 200.0, 211,  0.1, 0.2, 0.3, 50,213,90],
    [1,1, 200.0, 321,  -0.1, -0.2, -0.3, 40,213,90],
    [1,2, 200.0, 211,  0.2, 0.1, -0.1, 0.3,213,90],
    
    # Event 2
    [2,0, 300.0, 211,  0.3, 0.1, 0.2, 10,213,90],
    [2,1, 300.0, 321,  -0.2, -0.1, 0.1, 35,213,90],
    [2,2, 300.0, 211,  -0.3, 0.4, -0.2, 40,213,90],
    
    # Event 3
    [3,0, 400.0, 321,  0.5, 0.5, 0.4, 43,213,90],
    [3,1, 400.0, 211,  -0.3, -0.2, 0.1, 50,213,90],
    [3,2, 400.0, 321,  0.1, 0.2, -0.1, 51,213,90]
]
columns = ['ievent','iparticle', 'truth_energy', 'pid',  'px', 'py', 'pz', 'e', 'parent_pid1', 'parent_pid2']
data = pd.DataFrame(data, columns=columns)
data.to_csv("output_events_pion/events_1000.csv.zip", index=False)
data

,ievent,iparticle,truth_energy,pid,px,py,pz,e,parent_pid1,parent_pid2
0,1,0,200.0,211,0.1,0.2,0.3,50.0,213,90
1,1,1,200.0,321,-0.1,-0.2,-0.3,40.0,213,90
2,1,2,200.0,211,0.2,0.1,-0.1,0.3,213,90
3,2,0,300.0,211,0.3,0.1,0.2,10.0,213,90
4,2,1,300.0,321,-0.2,-0.1,0.1,35.0,213,90
5,2,2,300.0,211,-0.3,0.4,-0.2,40.0,213,90
6,3,0,400.0,321,0.5,0.5,0.4,43.0,213,90
7,3,1,400.0,211,-0.3,-0.2,0.1,50.0,213,90
8,3,2,400.0,321,0.1,0.2,-0.1,51.0,213,90


**ERROR: The input file is in-consistent: the particle masses are not what they should be. This leads to non-sensival results later**. 

## Decay Pions / Kaons

This function is taken from FORESEE to model the two body decay of $\pi^+$ / $K^+$ to $
\mu^+$ and $\nu_{\mu}$. It provides us with the produced particle's four momentum in lab frame


In [4]:
def twobody_decay(p0, m0, m1, m2):
    """
    Decays p0 -> p1 + p2 and returns the momenta of p1 and p2 in the lab frame.
    """
    phi = np.random.uniform(0, 2 * np.pi)
    costheta  = np.random.uniform(-1,1)
    
    zaxis = Vector3D(0, 0, 1)
    rotaxis = zaxis.cross(p0.vector).unit()
    rotangle = zaxis.angle(p0.vector)

    energy1 = (m0 * m0 + m1 * m1 - m2 * m2) / (2. * m0)
    energy2 = (m0 * m0 - m1 * m1 + m2 * m2) / (2. * m0)
    momentum1 = math.sqrt(energy1 * energy1 - m1 * m1)

    px1 = momentum1 * math.sqrt(1. - costheta * costheta) * np.cos(phi)
    py1 = momentum1 * math.sqrt(1. - costheta * costheta) * np.sin(phi)
    pz1 = momentum1 * costheta
    p1_rest = LorentzVector(-px1, -py1, -pz1, energy1)
    p2_rest = LorentzVector(px1, py1, pz1, energy2)

    if rotangle != 0:
        p1_rest = p1_rest.rotate(rotangle, rotaxis)
        p2_rest = p2_rest.rotate(rotangle, rotaxis)

    p1_lab = p1_rest.boost(-1. * p0.boostvector)
    p2_lab = p2_rest.boost(-1. * p0.boostvector)

    return p1_lab, p2_lab



To provide weight to the produced particle, we define the function `calculate_decay_probability()`. The probability of decay of pion is given by $1 - exp(-L/(c\tau\gamma))$. Here, L is the distance to detector, c$\tau$ is the lifetime of the meson and $\gamma = E / m$ is the boost factor.

In the following we use $m_\pi = 0.13957$ GeV, $m_K = 0.493$ GeV, $m_\mu = 0.10566$ GeV, $m_\nu=0$, $c\tau_\pi =7.8$ m and $c\tau_\pi =3.71$ m. We considered $L = 10 $m.

In [5]:
def decay_meson(px, py, pz, e, pid, detector_distance):
    """Simulate two-body decay of a pion (PID=211) or kaon (PID=321) into mu+ and neutrino."""
    
    # Define masses, lifetimes and BR
    mass_dict = {211: 0.13957, 321: 0.49367}
    branching_fraction = {211: 0.99, 321: 0.63}
    c_tau_values = {211: 7.8, 321: 3.71}
    muon_mass = 0.10566  
    neutrino_mass = 0.0   
    
    # obtain parent particle lifetime and mass
    c_tau = c_tau_values[pid]
    particle_mass = mass_dict[pid]
    
    # Particle 4-momentum
    p4_particle = LorentzVector(px, py, pz, e)
    
    # Decay probability calculation
    gamma = e / particle_mass
    mean_decay_length = c_tau * gamma
    decay_probability = (1 - np.exp(-detector_distance / mean_decay_length))*branching_fraction[pid]

    # Perform the decay
    muon, neutrino = twobody_decay(p4_particle, particle_mass, muon_mass, neutrino_mass)
    
    # return
    return muon, neutrino, decay_probability

The following code incorporates both pion and kaon decay. The code works the following way: 
- we always keep a copy of the original event with unit weight 
- for each pion/kaon, we will make a copies of the event and decay the particle an assign a weight accordig to the probability of decay.

In [6]:
def decay_and_flag(data, detector_distance):
    """Perform pion and kaon decay, generate event copies, and update event indices."""
    
    # iniitalzed updated data. 
    updated_data = []
    
    # loop over events in sample 
    for ievent, original_event in enumerate(data['ievent'].unique(), start=1):
        
        # Select all particles from this event
        event_data = data[data['ievent'] == original_event] 
        
        # Select pions (PID=211) and kaons (PID=321) with E > 30 GeV
        decay_candidates = event_data[(event_data['pid'].isin([211, 321])) & (event_data['e'] > 30)].index
        
        # Store the original event first (ievent remains the same)
        event_copy = event_data.copy()
        event_copy['weight'] = 1.0   
        event_copy['ievent'] = ievent   
        updated_data.append(event_copy)

        # Loop over each decay candidate
        for i, pid_idx in enumerate(decay_candidates, start=1):
            
            # make copy of event
            event_copy = event_data.copy()
            event_copy['ievent'] = ievent + i 
            
            # ge number of particles
            max_iparticle = event_data['iparticle'].max()

            # Get particle row 
            particle_row = event_copy.loc[pid_idx]
            #event_copy = event_copy.drop(pid_idx)   ### (dont drop it instead relace it by decayed muon)

            # Get PID
            pid = particle_row['pid']

            # Set weight = 1 for all non-decay particles
            event_copy['weight'] = 1.0

            # Perform decay into muon and neutrino
            muon, neutrino,decay_probability = decay_meson(
                particle_row['px'], particle_row['py'], 
                particle_row['pz'], particle_row['e'],
                pid, 
                detector_distance
            )
            
            # Create muon row
            muon_row = particle_row.copy()
            muon_row['pid'] = -13
            muon_row['iparticle'] = particle_row['iparticle']   #gets the same particle number as parent meson
            muon_row['px'], muon_row['py'], muon_row['pz'], muon_row['e'] = \
                muon.px, muon.py, muon.pz, muon.e
            muon_row['weight'] = decay_probability  # Set decay probability
            muon_row['parent_pid1'] = pid  # Parent ID = pion or kaon PID
            muon_row['parent_pid2'] = 90   # Second parent ID = 90
            
            # Replace the pion row with the muon row
            event_copy.loc[pid_idx] = muon_row
            
            # Create neutrino row
            neutrino_row = particle_row.copy()
            neutrino_row['pid'] = 14
            neutrino_row['iparticle'] = max_iparticle + 1  # Neutrino gets the next available iparticle number
            neutrino_row['px'], neutrino_row['py'], neutrino_row['pz'], neutrino_row['e'] = \
                neutrino.px, neutrino.py, neutrino.pz, neutrino.e
            neutrino_row['weight'] = decay_probability  # Set decay probability
            neutrino_row['parent_pid1'] = pid  # Parent ID = pion or kaon PID
            neutrino_row['parent_pid2'] = 90   # Second parent ID = 90

            # Add the neutrino at the end of the event copy
            event_copy = pd.concat([event_copy, pd.DataFrame([neutrino_row])], ignore_index=True)

            # Append the updated event to the final list
            updated_data.append(event_copy)
           
    return pd.concat(updated_data, ignore_index=True)

Here I comapre the input events file to the output after pion_decay. 

In [7]:
print("Input:")
data

Input:


,ievent,iparticle,truth_energy,pid,px,py,pz,e,parent_pid1,parent_pid2
0,1,0,200.0,211,0.1,0.2,0.3,50.0,213,90
1,1,1,200.0,321,-0.1,-0.2,-0.3,40.0,213,90
2,1,2,200.0,211,0.2,0.1,-0.1,0.3,213,90
3,2,0,300.0,211,0.3,0.1,0.2,10.0,213,90
4,2,1,300.0,321,-0.2,-0.1,0.1,35.0,213,90
5,2,2,300.0,211,-0.3,0.4,-0.2,40.0,213,90
6,3,0,400.0,321,0.5,0.5,0.4,43.0,213,90
7,3,1,400.0,211,-0.3,-0.2,0.1,50.0,213,90
8,3,2,400.0,321,0.1,0.2,-0.1,51.0,213,90


In [8]:
print("Output:")
decay_and_flag(data, detector_distance=10)

Output:


,ievent,iparticle,truth_energy,pid,px,py,pz,e,parent_pid1,parent_pid2,weight
0,1.0,0.0,200.0,211.0,0.100000,0.200000,0.300000,50.000000,213.0,90.0,1.000000
1,1.0,1.0,200.0,321.0,-0.100000,-0.200000,-0.300000,40.000000,213.0,90.0,1.000000
2,1.0,2.0,200.0,211.0,0.200000,0.100000,-0.100000,0.300000,213.0,90.0,1.000000
3,2.0,0.0,200.0,-13.0,-0.016490,-0.011206,-0.021083,0.109572,211.0,90.0,0.003537
4,2.0,1.0,200.0,321.0,-0.100000,-0.200000,-0.300000,40.000000,213.0,90.0,1.000000
5,2.0,2.0,200.0,211.0,0.200000,0.100000,-0.100000,0.300000,213.0,90.0,1.000000
6,2.0,3.0,200.0,14.0,0.016769,0.011764,0.021921,0.030002,211.0,90.0,0.003537
7,3.0,0.0,200.0,211.0,0.100000,0.200000,0.300000,50.000000,213.0,90.0,1.000000
8,3.0,1.0,200.0,-13.0,-0.131774,0.082951,0.174654,0.256736,321.0,90.0,0.020613
9,3.0,2.0,200.0,211.0,0.200000,0.100000,-0.100000,0.300000,213.0,90.0,1.000000


In the output there are now 10 events (original+copy). But the `ievent` variable is messed up. But that's no problem: This can be fixed wih the `fix_dataframe()` function that we introduced before. 

## Calculate Observables

Afterwards we can also save the observables. Note that 
Next, I need to calculate the observables.Now i have another column named weight with the weight of the decay products which I just extract from the modified input events file

In [9]:
def fix_dataframe(data):

    # Initialize the column 'ievent' with zeros
    data['ievent'] = 0
    # Variable to keep track of the increment for column 'ievent'
    counter = 0
    # Loop through the DataFrame rows
    for index, row in data.iterrows():
        if row['iparticle'] == 0: counter += 1
        data.at[index, 'ievent'] = counter
    #return
    return data

def calculate_observables_with_flags(data):
    """Calculate observables, incorporating the decay flag and weights."""
    
    # initialze events 
    events = []

    # loop over events
    grouped_data = data.groupby('ievent')
    for ievent, evt in grouped_data:

        # initialze observable and weight
        e_mu_minus, e_mu_plus, has_charm, e_em, e_visible, ptx, pty, ht = 0, 0, 0, 0, 0, 0, 0, 0
        pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus = 0, 0, 0, 0
        event_weight = 1.0

        # loop over particles
        for _, row in evt.iterrows():
            
            # get partilce information 
            truth_energy, pid, parent_pid1,  weight = row.iloc[2], row.iloc[3], row.iloc[8],  row['weight']
            px, py, pz, e = row.iloc[4], row.iloc[5], row.iloc[6], row.iloc[7]
            
            #get event weights
            event_weight = min(event_weight,weight)

            # check if it has charm
            if abs(parent_pid1) in {411, 421, 431, 4122, 15, 521, 511, 531, 443, 5122, 4232, 4112, 5232}: 
                has_charm = 1

            # muons
            if pid == 13: 
                if e > e_mu_minus:
                    e_mu_minus = e 
                    pt_mu_minus = np.sqrt(px**2 + py**2) 
                    phi_mu_minus = np.arctan2(py, px)
            elif pid == -13: 
                if e > e_mu_plus:
                    e_mu_plus = e 
                    pt_mu_plus = np.sqrt(px**2 + py**2) 
                    phi_mu_plus = np.arctan2(py, px)

            # EM particles
            if abs(pid) in {22, 11}: 
                e_em += e 
                
            # missing energy 
            if abs(pid) not in {12, 14, 16, 39}: 
                e_visible += e 
                ptx += px 
                pty += py 
                ht += np.sqrt(px**2 + py**2) 

        # ptmiss and phi 
        pt_mis = np.sqrt(ptx**2 + pty**2)
        phi_vis = np.arctan2(pty, ptx)
        phi_mis = np.arctan2(-pty, -ptx)

        # save event
        events.append([ievent, truth_energy, e_mu_minus, e_mu_plus, e_em, has_charm, e_visible, pt_mis, ht, pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus, phi_mis, phi_vis, event_weight])
        
    # save
    columns = ['ievent', 'truth_energy', 'e_mu_minus', 'e_mu_plus', 'e_em', 'has_charm', 'e_visible', 'pt_mis', 'ht', 'pt_mu_minus', 'pt_mu_plus', 'phi_mu_minus', 'phi_mu_plus', 'phi_mis', 'phi_vis', 'weight']
    observables = pd.DataFrame(events, columns=columns)

    return observables


## Event Loop

Here, I loop over all events, make copies using the `decay_and_flag()`, and calculate the observables. 

We start with our example files (`process =pion`, `energy=1000`)

In [10]:
energies = [1000]
processes = ["pion"]

for process in processes: 
    for energy in energies:
        
        #define filename
        filename = f"output_events_{process}/events_{str(energy)}.csv.zip"

        # check if file exists
        if not os.path.isfile(filename):
            print(f"File not found: {filename}")
            continue

        # logging
        print(process, energy)
        
        # read file
        data = pd.read_csv(filename)
        
        # Check if DataFrame is empty
        if data.empty:
            print(f"Data is empty for {filename}")
            continue
            
        # Check if the first row has a valid value
        if 0 not in data.index or np.isnan(data.loc[0, 'ievent']):
            #print(f"Fixing DataFrame for {filename}")
            data = fix_dataframe(data)
        
        # decay pions and fix frame again       
        data1 = decay_and_flag(data, detector_distance=10.0)
        data1 = fix_dataframe(data1)

        # calculate observables
        observables = calculate_observables_with_flags(data1)
        
        # save results
        csv_file = f"output_events_{process}/new_observables_{str(energy)}.csv.zip"
        observables.to_csv(csv_file, index=False, compression='zip')
        

pion 1000
